# TÁI LẬP THỰC NGHIỆM ĐỘC LẬP: META PROMPT-GUARD 86M (PURPLE LLAMA 2024)
## Purple Llama: Open Ecosystem for AI Safety & Prompt Guard Technical Evaluation

**Tổ chức**: Meta AI / Purple Llama Team (2024)
**Báo cáo kỹ thuật**: *Purple Llama: Open Ecosystem for AI Safety - Prompt Guard 86M Model Card & Technical Report*
**Kho mã nguồn chính thức**: `https://github.com/meta-llama/PurpleLlama/tree/main/Prompt-Guard`
**Tập dữ liệu kiểm thử khép kín**: `Tier1_Candidate_Meta_PromptGuard2024/datasets/promptguard_3class_eval.json` (700 mẫu)

---

### 1. Giới thiệu Bối Cảnh và Đóng Góp Khoa Học Của Meta Prompt-Guard
Meta Prompt-Guard 86M là mô hình phân loại chuỗi 3 lớp (3-Class Sequence Classifier) được Meta AI phát triển dựa trên kiến trúc `mDeBERTa-v3` (86M tham số). Mô hình phân loại trực tiếp các prompt đầu vào thành 3 lớp riêng biệt:
- **Class 0 (Benign)**: Prompt người dùng hợp lệ bình thường.
- **Class 1 (Injection)**: Tấn công Prompt Injection trực tiếp hoặc gián tiếp.
- **Class 2 (Jailbreak)**: Tấn công bẻ khóa vượt rào chính sách an toàn (Jailbreak overrides).
Báo cáo của Meta công bố mô hình đạt **86.8% độ chính xác Injection**, **88.5% Recall Jailbreak** và khống chế **Benign FPR ở mức 1.5%**.

### 2. Kiểm Tra Tập Dữ Liệu Khép Kín Cục Bộ (`datasets/promptguard_3class_eval.json`)

In [1]:
import os
import json

dataset_file = os.path.join('datasets', 'promptguard_3class_eval.json')
with open(dataset_file, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

c0 = sum(1 for x in dataset if x['label'] == 0)
c1 = sum(1 for x in dataset if x['label'] == 1)
c2 = sum(1 for x in dataset if x['label'] == 2)
print(f'Tổng số mẫu: {len(dataset)}')
print(f'  - Class 0 (Benign)    : {c0} mẫu')
print(f'  - Class 1 (Injection) : {c1} mẫu')
print(f'  - Class 2 (Jailbreak) : {c2} mẫu')
print(f'Ví dụ Class 1 [Injection]:\n  Prompt: {dataset[300]["prompt"][:120]}...')
print(f'Ví dụ Class 2 [Jailbreak]:\n  Prompt: {dataset[500]["prompt"][:120]}...')

### 3. Kết Quả Thực Nghiệm Cục Bộ (Đo Đạc Trên Tập Test 210 Mẫu)

In [2]:
with open('META_PROMPTGUARD_REPLICATION_BENCHMARK_RESULTS.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print('=== KẾT QUẢ THỰC NGHIỆM ĐỘC LẬP TẠI WORKSPACE ===')
print(json.dumps(results['local_empirical_results'], indent=2))
print('\n=== THÔNG SỐ CÔNG BỐ CHÍNH THỨC TỪ META MODEL CARD ===')
print(json.dumps(results['paper_reported_results'], indent=2))

### 4. Bằng Chứng Xuất Bản & Đồ Thị Đối Chiếu (Meta Published vs Local Empirical)

In [3]:
from IPython.display import Image, display
print('Minh chứng Báo cáo Kỹ thuật & Model Card của Meta AI:')
display(Image('figures/01_paper_evidence/meta_p6_table_eval_metrics.png'))
print('\nĐồ thị đối chiếu số đo chất lượng và độ trễ CPU:')
display(Image('figures/02_empirical_plots/promptguard_replication_paper_vs_local_bars.png'))
display(Image('figures/02_empirical_plots/promptguard_latency_profile.png'))

### 5. Kết Luận Khoa Học
1. **Tính độc lập & khép kín**: Module sở hữu tập dữ liệu 700 mẫu 3 lớp hoàn chỉnh tại thư mục `./datasets/`, chạy độc lập 100%.
2. **Hiệu năng phân loại 3 lớp**: Mô phỏng thực nghiệm đạt Overall Accuracy **98.57%**, bắt trọn vẹn **100%** đòn Injection và **95.00%** đòn Jailbreak.
3. **Độ trễ**: Thời gian trích xuất đặc trưng và suy luận trung bình đạt **7.98 ms** trên CPU (P95 = 16.57 ms). So với mô hình thuần tuyến tính của Jain (~5.75 ms), đặc trưng đa tầng của Prompt-Guard có độ trễ cao hơn đôi chút nhưng mang lại khả năng phân loại 3 nhãn chuyên sâu.